In [ ]:
import numpy as np
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt

from tqdm import tqdm 

from loka.data import load_breakthrough_states, save_breakthrough_states, load_forgetting_states, save_forgetting_states
from loka.models import BreakthroughCNN
from loka.training_methods import ForgettingScores
from loka.geometrical_methods import herding_order

import os
import pandas as pd
from dotenv import load_dotenv
import random

load_dotenv()
BASE_DIR = os.getenv("BASE_DIR")

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

X,Y = load_breakthrough_states(f"{BASE_DIR}/data/game_states/e.txt")

In [ ]:
herding_order = herding_order(X, 1, device)

In [ ]:
epochs = 200

pbar = tqdm(
    total=epochs,
    desc="training",
    bar_format="{l_bar}{bar} {percentage:3.0f}% [{elapsed}<{remaining}]",
)

X_t, Y_t = X, Y

torch.manual_seed(SEED)
model = BreakthroughCNN()
forgetter = ForgettingScores(model, seed=SEED)

forgetter.train(
    X = X_t,
    Y = Y_t,
    epochs = epochs,
    batch_size= 512,
    random_permuations=True,
    device=device,
    pbar=pbar,
    pbar_increment=1
)


In [ ]:
C = forgetter.getForgettingCounts()
C

In [ ]:
import random
import time

def make_forgettable_subset(E, Y, C, order, S_size, forgettable_size):
    forgettable_count = 0

    S = E[order][:S_size]
    S_labels = Y[order][:S_size]          # labels for S

    j = 0 # forgettable chosen

    forgattable_index_shuffle = list()

    for k,x in enumerate(C):
        if (x.item() != 0):
            forgattable_index_shuffle.append(k)

    random.shuffle(forgattable_index_shuffle)

    i = len(S)-1

    while forgettable_count < forgettable_size:
        oi = order[i]

        unforgettable = C[oi].item() == 0

        if unforgettable:
            k = forgattable_index_shuffle[j]
            S[i] = E[k]
            S_labels[i] = Y[k]            # swap the label with the event
            j += 1

        i -= 1
        forgettable_count += 1

    return S, S_labels        


S_ratio = 0.05
forgettable_size = 0.3

S_count = int(S_ratio*len(X))
forgettable_count = int(S_count * forgettable_size)

s0 = time.time()
X_S, Y_S = make_forgettable_subset(X, Y, C, herding_order, S_count, forgettable_count)
print(time.time()-s0)

In [ ]:
import numpy as np
import torch
import time

def make_forgettable_subset(
    E: torch.Tensor,
    Y: torch.Tensor,
    C: torch.Tensor,
    order: np.ndarray,
    S_size: int,
    forgettable_size: int,
) -> tuple[torch.Tensor, torch.Tensor]:
    assert forgettable_size <= S_size

    idx: torch.Tensor = torch.as_tensor(order[:S_size], device=C.device).long().clone()
    tail: torch.Tensor = idx[S_size - forgettable_size:]    # view into idx

    bad: torch.Tensor = C[tail] == 0                        # unforgettable slots in the tail
    n: int = int(bad.sum())

    mask: torch.Tensor = C != 0
    mask[idx] = False                                       # exclude events already in S
    pool: torch.Tensor = mask.nonzero().squeeze(1)
    pool = pool[torch.randperm(len(pool), device=pool.device)[:n]]

    tail[bad] = pool
    idx_cpu = idx.cpu().numpy()
    return E[idx_cpu], Y[idx_cpu]


S_ratio = 0.05
forgettable_size = 0.3

S_count = int(S_ratio*len(X))
forgettable_count = int(S_count * forgettable_size)

s0 = time.time()
X_S, Y_S = make_forgettable_subset(X, Y, C, herding_order, S_count, forgettable_count)
print(time.time()-s0)

In [ ]:
S_ratio = 0.05
forgettable_size = 0.3

S_count = int(S_ratio*len(X))
forgettable_count = int(S_count * forgettable_size)

X_S, Y_S = make_forgettable_subset(X, Y, C, herding_order, S_count, forgettable_count)